<a href="https://colab.research.google.com/github/rifqiiiz/UTS-Design-Web/blob/master/02_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Data Preparation & Feature Engineering **[L]**

Tahap ini menyiapkan dataset agar bersih, bebas *data leakage*, dan siap digunakan untuk pelatihan model klasik (Scikit-Learn) maupun model Deep Learning (`MLPModel` PyTorch).

In [5]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

# 1. Memuat ulang dataset mentah
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz"
df = pd.read_csv(url, compression="gzip")

# 2. Data Cleaning (Bab 5.1)
# - Menghapus baris duplikat
df = df.drop_duplicates().reset_index(drop=True)

# - Menangani outlier fisik pada fitur suhu ('temp' dalam Kelvin; 0 K tidak valid secara fisik)
df = df[df["temp"] > 200].reset_index(drop=True)

# - Mengisi missing values pada kolom 'holiday' (jika terbaca NaN) menjadi 'None'
df["holiday"] = df["holiday"].fillna("None")

print(f"Jumlah data setelah cleaning: {df.shape[0]} baris")

Jumlah data setelah cleaning: 48177 baris


In [6]:
# 3. Feature Engineering (Bab 5.2)
df["date_time"] = pd.to_datetime(df["date_time"])
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month

# Fitur domain: akhir pekan & jam sibuk (rush hour: 07-09 pagi dan 16-18 sore di hari kerja)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_rush_hour"] = (
    (df["is_weekend"] == 0) & (df["hour"].isin([7, 8, 9, 16, 17, 18]))
).astype(int)

# Representasi siklikal jam agar model memahami jam 23 berdekatan dengan jam 0
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

# Pembentukan label target multikelas
def categorize_traffic(volume):
    if volume < 2500:
        return "Lancar"
    elif volume < 5000:
        return "Padat"
    else:
        return "Macet"

df["target_label"] = df["traffic_volume"].apply(categorize_traffic)

# Pemilihan fitur (menghapus 'date_time', 'weather_description', dan 'traffic_volume' agar tidak bocor)
numeric_features = [
    "temp", "rain_1h", "snow_1h", "clouds_all",
    "hour_sin", "hour_cos", "day_of_week", "month",
    "is_weekend", "is_rush_hour"
]
categorical_features = ["holiday", "weather_main"]

X = df[numeric_features + categorical_features]
y_raw = df["target_label"]

# Encoding label target menjadi angka diskret (0, 1, 2)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
print("Mapping kelas target:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

Mapping kelas target: {'Lancar': 0, 'Macet': 1, 'Padat': 2}


In [7]:
# 4. Data Splitting (Bab 5.4)
# Aturan Emas: Test set (20%) disisihkan dan TIDAK disentuh hingga tahap evaluasi akhir [P]
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Membagi sisa 80% menjadi 60% Train (0.75 * 0.80) dan 20% Validation (0.25 * 0.80)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Total sampel   : {len(X)}")
print(f"Training set   : {len(X_train)} sampel ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set : {len(X_val)} sampel ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set       : {len(X_test)} sampel ({len(X_test)/len(X)*100:.1f}%)")

Total sampel   : 48177
Training set   : 28905 sampel (60.0%)
Validation set : 9636 sampel (20.0%)
Test set       : 9636 sampel (20.0%)


In [8]:
# 5. Data Transformation (Bab 5.3)
# StandardScaler untuk fitur numerik, OneHotEncoder untuk fitur kategorikal
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

# PENTING: fit_transform HANYA dilakukan pada X_train untuk mencegah data leakage!
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"Dimensi fitur setelah preprocessing: {X_train_processed.shape[1]} fitur")

# Menyimpan data hasil preprocessing dan objek preprocessor untuk tahap Modeling [M+O]
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

np.savez(
    "../data/processed/traffic_splits.npz",
    X_train=X_train_processed, y_train=y_train,
    X_val=X_val_processed, y_val=y_val,
    X_test=X_test_processed, y_test=y_test
)
joblib.dump(preprocessor, "../models/preprocessor.joblib")
joblib.dump(label_encoder, "../models/label_encoder.joblib")

print("Data preparation selesai! Artefak tersimpan di folder data/processed/ dan models/.")

Dimensi fitur setelah preprocessing: 33 fitur
Data preparation selesai! Artefak tersimpan di folder data/processed/ dan models/.
